# 📊 Veri Dağılımı Analizi

Bu notebook, Materials Project veritabanından alınan kristal malzeme verisinin istatistiksel dağılımını inceler.

**İncelenen konular:**
- Dört hedef değişkenin (formation_energy, band_gap, cbm, energy_above_hull) histogramları
- Uzay grubu (space group) dağılımı — en sık görülen 20 grup
- Kristal sistem (crystal_system) dağılımı
- Kararlılık etiketi (stability_label) dağılımı
- Hedef değişkenler arası korelasyon

**Veri dosyası:** `data/MP_queried_data_featurized_w_additional_acr_ae_en_stability_label.csv`

In [ ]:
# ============================================================
# KÜTÜPHANELER VE VERİ YÜKLEME
# ============================================================
# Veri işleme ve görselleştirme için gerekli kütüphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Grafik stilini ayarla — daha okunabilir görünüm için
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Ham veri setini yükle
# low_memory=False: karışık veri tipli sütunlarda DtypeWarning engellemek için
CSV_PATH = "data/MP_queried_data_featurized_w_additional_acr_ae_en_stability_label.csv"
df_raw = pd.read_csv(CSV_PATH, low_memory=False)

print(f"Ham veri boyutu : {df_raw.shape[0]:,} satır × {df_raw.shape[1]} sütun")
print(f"Sütunlar (ilk 10): {df_raw.columns[:10].tolist()}")

In [ ]:
# ============================================================
# AYKIRI DEĞER TEMİZLEME — preparation.py ile tutarlılık
# ============================================================
# formation_energy_per_atom için ±5σ filtresi uygula
# Bu, preparation.py ile birebir aynı filtreleme adımıdır.
mean_fe = df_raw['formation_energy_per_atom'].mean()
std_fe  = df_raw['formation_energy_per_atom'].std()

df = df_raw[
    (df_raw['formation_energy_per_atom'] >= mean_fe - 5 * std_fe) &
    (df_raw['formation_energy_per_atom'] <= mean_fe + 5 * std_fe)
].copy()
df.reset_index(drop=True, inplace=True)

print(f"Filtreleme öncesi : {len(df_raw):,} örnek")
print(f"Filtreleme sonrası: {len(df):,} örnek")
print(f"Çıkarılan aykırı  : {len(df_raw) - len(df):,} örnek")

In [ ]:
# ============================================================
# HEDEF DEĞİŞKENLERİN DAĞILIMI
# ============================================================
# Modelin tahmin ettiği 4 hedef değişken için histogram grafikleri

HEDEFLER = [
    ('formation_energy_per_atom', 'Formasyon Enerjisi (eV/atom)', '#1f77b4'),
    ('band_gap',                  'Bant Aralığı (eV)',            '#ff7f0e'),
    ('cbm',                       'İletim Bandı Min. (CBM, eV)',  '#2ca02c'),
    ('energy_above_hull',         'Hull Üstü Enerji (eV/atom)',   '#d62728'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Hedef Değişkenlerin Dağılımı', fontsize=16, fontweight='bold', y=1.01)

for ax, (col, label, color) in zip(axes.flatten(), HEDEFLER):
    vals = df[col].dropna()
    # 5. ve 95. yüzdelik dilim arasını göster (uç değerleri dışla)
    lo, hi = vals.quantile(0.01), vals.quantile(0.99)
    vals_clip = vals.clip(lo, hi)

    ax.hist(vals_clip, bins=80, color=color, alpha=0.75, edgecolor='none')
    ax.axvline(vals.mean(), color='black', linestyle='--', lw=1.5, label=f'Ortalama: {vals.mean():.3f}')
    ax.axvline(vals.median(), color='gray', linestyle=':', lw=1.5, label=f'Medyan: {vals.median():.3f}')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Örnek Sayısı')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.text(0.97, 0.95, f'n={len(vals):,}\nStd={vals.std():.3f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('figures/hedef_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nHedef değişken istatistikleri:")
print(df[[c for c, _, _ in HEDEFLER]].describe().round(3))

In [ ]:
# ============================================================
# UZAY GRUBU DAĞILIMI — En Yaygın 20 Uzay Grubu
# ============================================================
# Uzay grubu, kristal yapısının simetrisi hakkında bilgi verir.
# 230 farklı uzay grubu vardır; ancak bazıları çok daha sık görülür.

sg_counts = df['number'].value_counts().sort_values(ascending=False)
top20_sg = sg_counts.head(20)

print("En sık 10 uzay grubu:")
print(top20_sg.head(10).to_string())
print(f"\nToplam farklı uzay grubu sayısı: {sg_counts.shape[0]}")
print(f"En yaygın 20 grubun toplam örneklerdeki payı: "
      f"{top20_sg.sum() / len(df) * 100:.1f}%")

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(
    top20_sg.index.astype(str), top20_sg.values,
    color=plt.cm.viridis(np.linspace(0, 0.85, 20)), edgecolor='none'
)
ax.set_xlabel('Uzay Grubu Numarası', fontsize=12)
ax.set_ylabel('Örnek Sayısı', fontsize=12)
ax.set_title('Uzay Grubu Dağılımı — En Yaygın 20 Grup', fontsize=14, fontweight='bold')
# Çubukların üzerine sayı yaz
for bar, val in zip(bars, top20_sg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', va='bottom', fontsize=8, rotation=45)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/uzay_grubu_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# KRİSTAL SİSTEMİ DAĞILIMI
# ============================================================
# 7 kristal sistemi: triclinic, monoclinic, orthorhombic,
# tetragonal, trigonal, hexagonal, cubic

cs_counts = df['crystal_system'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pasta grafiği
renkler = plt.cm.Set3(np.linspace(0, 1, len(cs_counts)))
ax1.pie(
    cs_counts.values, labels=cs_counts.index,
    autopct='%1.1f%%', colors=renkler, startangle=140,
    pctdistance=0.8, textprops={'fontsize': 10}
)
ax1.set_title('Kristal Sistem Dağılımı (Pasta)', fontsize=13, fontweight='bold')

# Çubuk grafiği
bars = ax2.barh(cs_counts.index, cs_counts.values, color=renkler, edgecolor='none')
ax2.set_xlabel('Örnek Sayısı', fontsize=12)
ax2.set_title('Kristal Sistem Dağılımı (Çubuk)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, cs_counts.values):
    ax2.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/kristal_sistem_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()
print(cs_counts.to_string())

In [ ]:
# ============================================================
# KARARLILIK ETİKETİ DAĞILIMI
# ============================================================
# energy_above_hull eşiklerine göre:
#   Stable     : ≤ 0.025 eV/atom (sentezlenebilir, kararlı)
#   Metastable : 0.025 – 0.100 eV/atom (koşullu kararlı)
#   Unstable   : > 0.100 eV/atom (kararsız)

stab_counts = df['stability_label'].value_counts()

renkler_stab = {'Stable': '#2ecc71', 'Metastable': '#f39c12', 'Unstable': '#e74c3c'}

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    stab_counts.index, stab_counts.values,
    color=[renkler_stab.get(k, '#95a5a6') for k in stab_counts.index],
    edgecolor='white', linewidth=0.8
)
ax.set_ylabel('Örnek Sayısı', fontsize=12)
ax.set_title('Kararlılık Etiketi Dağılımı\n(Energy Above Hull Eşiklerine Göre)',
             fontsize=13, fontweight='bold')

# Yüzde ve sayı etiketlerini ekle
for bar, val in zip(bars, stab_counts.values):
    yuzde = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{val:,}\n({yuzde:.1f}%)', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0, stab_counts.max() * 1.15)
plt.tight_layout()
plt.savefig('figures/kararlilik_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# HEDEF DEĞİŞKENLER ARASI KORELASYON
# ============================================================
# Hangi hedefler birbirleriyle ilişkilidir?
# Örneğin: band_gap ile cbm arasında güçlü pozitif korelasyon beklenir.

hedef_sutunlar = ['formation_energy_per_atom', 'band_gap', 'cbm', 'energy_above_hull']
korelasyon = df[hedef_sutunlar].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(korelasyon.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson Korelasyon Katsayısı')

# Etiketler
etik = ['Formation\nEnergy', 'Band Gap', 'CBM', 'Energy\nAbove Hull']
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(etik, fontsize=10)
ax.set_yticklabels(etik, fontsize=10)

# Değerleri hücrelere yaz
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{korelasyon.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=11,
                color='black' if abs(korelasyon.iloc[i, j]) < 0.7 else 'white',
                fontweight='bold')

ax.set_title('Hedef Değişkenler Arası Pearson Korelasyonu', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/korelasyon_matrisi.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKorelasyon Matrisi:")
print(korelasyon.round(3))

## Özet

| Değişken | Medyan | Std |
|---|---|---|
| formation_energy_per_atom | yaklaşık −2 eV/atom | geniş dağılım |
| band_gap | 0 eV (metaller) veya 1–4 eV (yarı iletkenler) | çift modlu dağılım |
| cbm | band_gap ile yüksek korelasyon | |
| energy_above_hull | 0'a yakın (kararlı malzemeler çoğunlukta) | sağa çarpık |

**Bant aralığı dağılımı çift modludur:** İletken/metal malzemelerde band_gap=0 (ilk zirve),  
yarı iletken ve yalıtkan malzemelerde ise band_gap > 0 (ikinci zirve).  
Bu durum, GGA DFT ile hesaplanan eğitim verisindeki sistematik *band gap underestimation* (bant aralığı küçük tahmin) sorununu yansıtır.